# Customer Support Copilot — Google Colab Notebook

This notebook is the **Colab-compatible entry point**. It will:
1. Install dependencies (pinned versions from `requirements.txt`)
2. Clone / unpack the `project/` tree into `/content/project`
3. Mount Google Drive (optional — for dataset + checkpoint persistence)
4. Fetch the MultiDoc2Dial dataset and build chunks / splits
5. (Optional) train the bi-encoder and DoRA adapters
6. Run baseline vs full evaluation, write JSON results, print the delta
7. Show an interactive demo cell

**Hardware**: enable a GPU runtime (`Runtime → Change runtime type → T4 GPU`). All steps also work on CPU, just slower.

## 1. Environment check

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)
print('Python:', sys.version.split()[0])
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('Torch not yet installed — will install in the next cell.')

## 2. Install dependencies
First cell takes ~2 minutes in Colab; subsequent cells reuse the installed packages.

In [ ]:
# Minimal install — all the versions the modules have been tested against.
!pip install -q 'transformers>=4.40,<5.0' 'accelerate>=0.28,<1.0' 'peft>=0.9' \
              'sentence-transformers>=2.6' 'faiss-cpu>=1.7.4' \
              'pydantic>=2.6' 'pyyaml>=6.0' 'nltk>=3.8' 'rouge-score>=0.1.2' \
              'fastapi>=0.110' 'uvicorn>=0.27' 'beautifulsoup4>=4.12' 'pytest>=8.0'

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ dependencies installed')

## 3. Unpack the project tree

If you uploaded `CustomerSupportCopilot_restructured.zip` to the Colab session, run the first path below. Otherwise the second path unpacks it from your Drive.

In [ ]:
import os, subprocess, zipfile

PROJECT_DIR = '/content/project'
REPO_URL    = 'https://github.com/Manya123-web/CustomerSupport_Copilot.git'
ZIP_PATH    = None

# --- Option A: zip is already on the Colab VM (legacy path) ---
for p in ['/content/CustomerSupportCopilot_restructured.zip',
          '/content/drive/MyDrive/CustomerSupportCopilot_restructured.zip']:
    if os.path.exists(p):
        ZIP_PATH = p
        break

# --- Option B: mount Drive and locate the zip there ---
if ZIP_PATH is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        for p in ['/content/drive/MyDrive/CustomerSupportCopilot_restructured.zip']:
            if os.path.exists(p):
                ZIP_PATH = p
                break
    except Exception as e:
        print('Drive not mounted:', e)

if ZIP_PATH:
    print('Unpacking', ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content')
    assert os.path.isdir(PROJECT_DIR), 'Unpacked but project/ not found'
else:
    # --- Option C (fallback): clone the repo directly from GitHub ---
    # This is the path the "Open in Colab" badge relies on: the notebook
    # is served from GitHub with no zip on the VM, so we clone instead.
    if not os.path.isdir(PROJECT_DIR):
        print(f'No zip found — cloning {REPO_URL} into {PROJECT_DIR}')
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, PROJECT_DIR],
                            capture_output=True, text=True)
        if r.returncode != 0:
            print('git clone failed:')
            print(r.stdout)
            print(r.stderr)
            print('⚠️  Fall back to: upload CustomerSupportCopilot_restructured.zip '
                   '(File → Upload) and re-run this cell.')
        else:
            print('✅ cloned into', PROJECT_DIR)
    else:
        print(f'{PROJECT_DIR} already exists — skipping clone.')

In [ ]:
# Put the project on sys.path so `import utils.metrics` etc. work.
import sys, os
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('cwd:', os.getcwd())
print('tree (top 2 levels):')
for root, dirs, files in os.walk('.'):
    depth = root.count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + '/')

## 4. Get the dataset

Two options. Pick whichever is easier for you.

- **A.** Use the Hugging Face `IBM/multidoc2dial` dataset (auto-downloads, no manual step).
- **B.** Drop a pre-prepared `multidoc2dial_raw.json` into `data/raw/`.

The `prepare_dataset()` helper below handles both.

In [ ]:
import json, os, random, re
from utils.config import load_config

cfg = load_config('config/config.yaml')
raw_path = cfg['data']['raw_path']
os.makedirs(os.path.dirname(raw_path), exist_ok=True)

def prepare_dataset():
    if os.path.exists(raw_path):
        with open(raw_path) as f:
            docs = json.load(f)
        print(f'Found existing {raw_path} with {len(docs)} docs')
        return docs
    # Try Hugging Face
    try:
        from datasets import load_dataset
        ds = load_dataset('IBM/multidoc2dial', 'multidoc2dial_doc', split='train')
        docs = [{'doc_id': r['doc_id'], 'text': r['doc_text']} for r in ds]
        with open(raw_path, 'w') as f:
            json.dump(docs, f)
        print(f'Fetched {len(docs)} docs via Hugging Face → {raw_path}')
        return docs
    except Exception as e:
        print('HF fetch failed:', e)
        return None

docs = prepare_dataset()

In [ ]:
# Build chunks + document-level 80/20 split
# Accepts either 'text' (HF-normalised) or 'doc_text' (raw MultiDoc2Dial dump)
if docs:
    size, overlap = cfg['data']['chunk_size'], cfg['data']['chunk_overlap']
    chunks = []
    for d in docs:
        text = d.get('text') or d.get('doc_text') or ''
        if not text:
            continue
        words = re.sub(r'\s+', ' ', text).split()
        step  = size - overlap
        for i in range(0, len(words), step):
            t = ' '.join(words[i:i+size])
            if len(t.split()) < 20:
                continue
            chunks.append({'chunk_id': f"{d['doc_id']}__{i//step}",
                            'doc_id':   d['doc_id'],
                            'text':     t})
    random.Random(cfg['seed']).shuffle(chunks)
    doc_ids = sorted({c['doc_id'] for c in chunks})
    random.Random(cfg['seed']).shuffle(doc_ids)
    cut = int(len(doc_ids) * cfg['data']['train_ratio'])
    tr_ids, te_ids = set(doc_ids[:cut]), set(doc_ids[cut:])
    train = [c for c in chunks if c['doc_id'] in tr_ids]
    test  = [c for c in chunks if c['doc_id'] in te_ids]
    for p in [cfg['data']['processed_path'], cfg['data']['train_split'], cfg['data']['test_split']]:
        os.makedirs(os.path.dirname(p), exist_ok=True)
    json.dump(chunks, open(cfg['data']['processed_path'], 'w'))
    json.dump(train,  open(cfg['data']['train_split'], 'w'))
    json.dump(test,   open(cfg['data']['test_split'], 'w'))
    print(f'✅ {len(chunks)} chunks  train={len(train)}  test={len(test)}')

## 5. Quick offline tests (no model downloads)

Sanity-check the codebase before spending GPU time.

In [ ]:
!python -m pytest tests/test_metrics.py tests/test_schema.py tests/test_agent_routing.py tests/test_policy_db.py -v

## 6. Train (OPTIONAL — skip if you want to test on base models)

`--stage biencoder` is ~5 minutes on T4. `--stage dora` is ~15–20 minutes on T4.

In [ ]:
# Train Component A (bi-encoder)
# !python -m training.train --stage biencoder --config config/full.yaml

In [ ]:
# Train Components C + D (DoRA + DPO)
# !python -m training.train --stage dora --config config/full.yaml

## 7. Run evaluation (baseline then full), then compare

In [ ]:
!python -m training.evaluation --config config/baseline.yaml

In [ ]:
!python -m training.evaluation --config config/full.yaml

In [ ]:
!python -m training.evaluation --compare

## 8. Interactive demo

Below: type a query, see tool trace + grounded answer.

In [ ]:
from utils.config import load_config
from training.evaluation import build_system
from training.agent import run_agent

sys_full = build_system(load_config('config/full.yaml'))

def ask(q):
    r = run_agent(q, retriever=sys_full['retriever'],
                     llm_fn=sys_full['llm_fn'],
                     reranker=sys_full['reranker'],
                     cfg=sys_full['cfg'])
    print('Q:', q)
    print('A:', r.final_answer)
    print('   grounding =', round(r.grounding_score, 3),
          '  id =', round(r.id_score, 3),
          '  latency =', round(r.latency_ms, 1), 'ms')
    if r.ticket:
        print('   ticket:', r.ticket.get('ticket_id'))
    print()

ask('What documents do I need to apply for Social Security retirement benefits?')
ask("I can't log into my account")
ask('How do I appeal a denied disability claim?')

## 9. Overfitting / underfitting / leakage diagnostics

Prints a single report with train vs test retrieval, fusion weight sanity, router generalisation, and cache-leak scan.

In [ ]:
from training.diagnostics import run_diagnostics
_ = run_diagnostics('config/full.yaml')

## 10. Run the chatbot on 30 curated demo questions

8 standard + 8 paraphrased + 7 account-issue + 7 out-of-scope. Shows how it behaves on each class: standard queries should answer with citations, account issues should ticket, out-of-scope should escalate via low ID score.

In [ ]:
from training.demo_questions import all_demo_questions

current_cat = None
for i, item in enumerate(all_demo_questions(), 1):
    if item['category'] != current_cat:
        current_cat = item['category']
        print(f"\n{'=' * 70}\n=== {current_cat} ===\n{'=' * 70}")
    r = run_agent(item['query'], retriever=sys_full['retriever'],
                     llm_fn=sys_full['llm_fn'],
                     reranker=sys_full['reranker'],
                     cfg=sys_full['cfg'])
    ticket_mark = f" [ticket:{r.ticket['ticket_id']}]" if r.ticket else ''
    print(f"\n{i:2d}. Q: {item['query']}")
    print(f"    A: {r.final_answer[:220]}{'...' if len(r.final_answer) > 220 else ''}")
    print(f"       grounding={r.grounding_score:.2f}  id={r.id_score:.2f}  "
          f"latency={r.latency_ms:.0f}ms{ticket_mark}")